In [1]:
from ase.build import graphene_nanoribbon, add_adsorbate, molecule
from ase.optimize import LBFGS, BFGS
from ase.constraints import FixAtoms
from fairchem.core import pretrained_mlip, FAIRChemCalculator
import numpy as np
from ase.visualize import view
import nglview as nv
from ase.io import write

W0711 17:18:57.858000 65595 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [3]:
path = "uma-s-1p1.pt"
predictor = pretrained_mlip.load_predict_unit(path, device="cpu")
calc = FAIRChemCalculator(predictor, task_name="oc20")

In [13]:
from ase.io import read

slab = read('B-N-doped.vasp', index=-1, format="vasp")
print(slab.get_pbc())
slab.center(vacuum=12.885,
            axis=2
           )
cell_size = [[20.6665058136, 0.0, 0.0], [10.3246228797, 17.8548927014, 0.0], [0.0, 0.0, 25.777]]
slab.set_cell(cell_size)

[ True  True  True]


In [14]:
nv.show_ase(slab, gui=True)

NGLWidget()

In [15]:
len(slab)

108

In [16]:
params = slab.cell.cellpar()
print(f"a: {params[0]:.3f}, b: {params[1]:.3f}, c: {params[2]:.3f}")
print(f"Alpha: {params[3]}, Beta: {params[4]}, Gamma: {params[5]}")
slab.get_cell()


a: 20.667, b: 20.625, c: 25.777
Alpha: 90.0, Beta: 90.0, Gamma: 59.961277007893294


Cell([[20.6665058136, 0.0, 0.0], [10.3246228797, 17.8548927014, 0.0], [0.0, 0.0, 25.777]])

In [17]:
slab.calc = calc
opt_slab = BFGS(slab)  # Relax clean slab if not already
opt_slab.run(fmax=0.05, steps=50)  # Shorter for clean slab
e_slab = slab.get_potential_energy()
print(f"Clean slab energy: {e_slab:.3f} eV")

      Step     Time          Energy          fmax
BFGS:    0 17:21:52     -885.889165        0.433208
BFGS:    1 17:21:52     -885.893574        0.597625
BFGS:    2 17:21:53     -885.924164        0.102544
BFGS:    3 17:21:54     -885.925398        0.085692
BFGS:    4 17:21:55     -885.925937        0.068606
BFGS:    5 17:21:56     -885.926382        0.038590
Clean slab energy: -885.926 eV


In [18]:
nv.show_ase(slab, gui=True)

NGLWidget()

In [19]:
selected_indices = [106, 107, 101, 56, 83, 92]

In [20]:
ads_ref = molecule("O2")
ads_ref.set_cell([20, 20, 20])
ads_ref.center()
ads_ref.calc = calc
opt_ads = BFGS(ads_ref)
opt_ads.run(fmax=0.05, steps=20)
e_ads = ads_ref.get_potential_energy()
print(f"Isolated O2 energy: {e_ads:.3f} eV")

      Step     Time          Energy          fmax
BFGS:    0 17:21:58       -8.469473        0.598350
BFGS:    1 17:21:58       -8.470468        0.504472
BFGS:    2 17:21:58       -8.472360        0.016663
Isolated O2 energy: -8.472 eV


In [21]:
atom=slab[106]
target_position = atom.position[:2]
adsorbate = molecule("O2")
adsorbate.rotate(90, 'x')
add_adsorbate(slab, adsorbate, height=2.68, position=target_position)

print(f"e_slab {e_slab} \t e_ads {e_ads}")


nv.show_ase(slab, gui=True)



e_slab -885.9263822807792 	 e_ads -8.472360004867877


NGLWidget()

In [22]:
opt = BFGS(slab)
opt.run(fmax=0.05, steps=200)




e_slab_ads = slab.get_potential_energy()
adsorption_energy = e_slab_ads - (e_slab + e_ads)

      Step     Time          Energy          fmax
BFGS:    0 17:22:14     -894.478583        1.276247
BFGS:    1 17:22:15     -894.496816        0.503962
BFGS:    2 17:22:16     -894.503414        0.488738
BFGS:    3 17:22:17     -894.576481        0.648162
BFGS:    4 17:22:17     -894.603875        0.630126
BFGS:    5 17:22:18     -894.632311        0.372466
BFGS:    6 17:22:19     -894.641267        0.337358
BFGS:    7 17:22:20     -894.649718        0.244230
BFGS:    8 17:22:21     -894.656030        0.257980
BFGS:    9 17:22:21     -894.659690        0.171168
BFGS:   10 17:22:22     -894.662839        0.202399
BFGS:   11 17:22:23     -894.665299        0.184688
BFGS:   12 17:22:24     -894.668510        0.198105
BFGS:   13 17:22:25     -894.675961        0.244078
BFGS:   14 17:22:25     -894.681782        0.276709
BFGS:   15 17:22:26     -894.692322        0.310999
BFGS:   16 17:22:27     -894.713834        0.337318
BFGS:   17 17:22:28     -894.732631        0.493375
BFGS:   18 17:

In [23]:
print(f"Atom({atom.symbol}): Adsorption energy = {adsorption_energy:.3f} eV")

Atom(B): Adsorption energy = -0.426 eV


In [24]:
nv.show_ase(slab, gui=True)

NGLWidget()

In [15]:
atom=slab[109]
target_position = atom.position[:2]
adsorbate = molecule("H2")
adsorbate.rotate(90, 'x')
add_adsorbate(slab, adsorbate, height=2.68, position=target_position)

# slab.calc = calc
# e_slab = slab.get_potential_energy()

print(f"e_slab {e_slab} \t e_ads {e_ads}")


nv.show_ase(slab, gui=True)

e_slab -885.9263980164054 	 e_ads -8.472360004867877


NGLWidget()

In [16]:
opt = BFGS(slab)
opt.run(fmax=0.05, steps=200)




e_slab_ads = slab.get_potential_energy()
adsorption_energy = e_slab_ads - (e_slab + e_ads)

      Step     Time          Energy          fmax
BFGS:    0 12:03:09     -769.235341      132.407852
BFGS:    1 12:03:10     -786.873800      584.927002
BFGS:    2 12:03:10     -862.006013      249.851089
BFGS:    3 12:03:11     -882.682235      127.144844
BFGS:    4 12:03:12     -896.845732       24.392809
BFGS:    5 12:03:13     -899.386937        4.792286
BFGS:    6 12:03:14     -900.059674        3.756837
BFGS:    7 12:03:14     -899.982626        7.378989
BFGS:    8 12:03:15     -900.435618        2.481232
BFGS:    9 12:03:16     -900.533756        2.269266
BFGS:   10 12:03:17     -900.973978        1.481468
BFGS:   11 12:03:18     -901.021550        2.420198
BFGS:   12 12:03:19     -901.107010        1.524573
BFGS:   13 12:03:19     -901.180894        0.788298
BFGS:   14 12:03:20     -901.227555        1.629956
BFGS:   15 12:03:21     -901.289034        2.044518
BFGS:   16 12:03:22     -901.346550        1.305530
BFGS:   17 12:03:23     -901.382551        0.952171
BFGS:   18 12:

In [17]:
nv.show_ase(slab, gui=True)

NGLWidget()

In [341]:
from ase.io import Trajectory
traj = Trajectory('trajFiles/over_N_BN_sheet.traj', 'w')
traj.write(slab)
write('trajFiles/over_N_BN_sheet.xyz', slab)


In [ ]:
anim = nv.show_ase(slab, gui=True)
anim.clear_representations()
anim.add_representation("ball+stick", selection="all", radius=0.3)
anim

In [39]:
anim.download_image(filename="trajFiles/slab_current_frame.png")